# Проект "Игровая индустрия начала XXI века"

## Описание проекта

### Цели и задачи проекта
В этом проекте будут изучены данные из датасета `new_games.csv`, который содержит информацию о продажах игр разных жанров и платформ, а также о пользовательских и экспертных оценках.  
Задача состоит в том, чтобы сделать обзор игровых платформ, изучить объёмы продаж игр разных жанров и региональные предпочтения игроков с 2000 по 2013 год. Отдельный акцент будет сделан на играх жанра RPG — так называют компьютерные ролевые игры, в которых игроки управляют персонажами.  
Исторические данные собраны из открытых источников.

### Описание данных
В проекте будут использованы данные датасета `new_games.csv` с таким описанием:
- `Name` — название игры.
- `Platform` — название платформы.
- `Year of Release` — год выпуска игры.
- `Genre` — жанр игры.
- `NA sales` — продажи в Северной Америке (в миллионах проданных копий).
- `EU sales` — продажи в Европе (в миллионах проданных копий).
- `JP sales` — продажи в Японии (в миллионах проданных копий).
- `Other sales` — продажи в других странах (в миллионах проданных копий).
- `Critic Score` — оценка критиков (от 0 до 100).
- `User Score` — оценка пользователей (от 0 до 10).
- `Rating` — рейтинг организации ESRB (англ. Entertainment Software Rating Board). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию.

### Содержимое проекта
1. Загрузка и знакомство с данными
2. Предобработка данных
3. Анализ данных
4. Выводы

## Загрузка и знакомство с данными

In [1]:
# Импортируем библиотеку pandas
import pandas as pd

In [2]:
# Выгружаем данные из датасета new_games.csv в датафрейм df
try:
    df = pd.read_csv(r'S:\project_1\new_games.csv')
except FileNotFoundError:
    df = pd.read_csv('/datasets/new_games.csv')

In [3]:
# Выводим информацию о датафрейме
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


In [4]:
# Выводим первые строки датафрейма на экран
df.head()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


Датасет `new_games.csv` содержит 11 столбцов и 16956 строк. В 6 из 11 столбцов встречаются пропуски, но ощутимо много их только в 3 столбцах.

Данные соответствуют описанию, заголовки отражают содержание столбцов, внешний вид заголовков следует изменить для более удобной работы с данными.

**Изучим типы данных и их корректность:**
- **Строковые данные (object).** Семь столбцов имеют тип данных `object`:
    - `Name` содержит уникальную строковую информацию. Здесь тип данных `object` подходит.
    - `Platform`, `Genre`, `Rating` хранят текстовые данные, но их можно рассматривать как категориальные признаки. В этом случае желательно использовать тип `category`, чтобы улучшить производительность и оптимизировать память, если набор значений ограничен и заведомо известен.
    - `EU sales`, `JP sales` содержат продажи в миллионах копий, с двумя знаками после запятой. Здесь необходим тип данных `float32` для выполнения расчетов. Заданной точности будет достаточно.
    - `User Score` содержит пользовательские оценки от 0 до 10, с одним знаком после запятой. Подойдет тип данных `float32`.
- **Числовые значения с плавающей запятой (float64).** Четыре столбца имеют тип данных `float64`:
    - `Year of Release` хранит год выпуска игры. В этом случае оптимально использовать `int16`, так как нам не потребуется проводить сложные операции с датой. При этом данный тип данных экономит память, позволяет быстрее и проще проводить фильтрацию.
    -  `NA sales`, `Other sales` - аналогично данным о продажах выше следует понизить разрядность до `float32`.
    -  `Critic Score` содержит оценки критиков от 0 до 100, с одним знаком после запятой. Можно понизить разрядность до `float32`.

## Предобработка данных

### Изменение названий столбцов

In [5]:
# Выведим на экран названия всех столбцов датафрейма
df.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

Стиль названий столбцов не соответствует стандарту в Python, приведем их в стилю snake_case

In [6]:
# Приводим названия столбцов к стилю snake_case
df = df.rename(columns={'Name': 'name',
                        'Platform': 'platform',
                        'Year of Release': 'year_released',
                        'Genre': 'genre',
                        'NA sales': 'na_sales',
                        'EU sales': 'eu_sales',
                        'JP sales': 'jp_sales',
                        'Other sales': 'other_sales',
                        'Critic Score': 'critic_score',
                        'User Score': 'user_score',
                        'Rating': 'rating'})

### Преобразование типов данных

In [7]:
# Преобразуем данные в категориальный тип
df[['platform', 'genre', 'rating']] = df[['platform', 'genre', 'rating']].astype('category', errors='ignore')

In [8]:
# Преобразуем данные в тип 'float32'
conv = ['na_sales', 'eu_sales', 'jp_sales', 'other_sales', 'critic_score', 'user_score']
df[conv] = df[conv].apply(pd.to_numeric, errors='coerce')
df[conv] = df[conv].astype('float32')

In [9]:
# Преобразуем столбец с годом релиза в тип данных 'Int16'
df['year_released'] = df['year_released'].astype('Int16')

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   name           16954 non-null  object  
 1   platform       16956 non-null  category
 2   year_released  16681 non-null  Int16   
 3   genre          16954 non-null  category
 4   na_sales       16956 non-null  float32 
 5   eu_sales       16950 non-null  float32 
 6   jp_sales       16952 non-null  float32 
 7   other_sales    16956 non-null  float32 
 8   critic_score   8242 non-null   float32 
 9   user_score     7688 non-null   float32 
 10  rating         10085 non-null  category
dtypes: Int16(1), category(3), float32(6), object(1)
memory usage: 631.7+ KB


In [11]:
# Зафиксируем кол-во строк до обработки
initial_rows = df.shape[0]
initial_rows

16956

### Работа с пропусками

In [12]:
# Выводим количество пропущенных строк в датафрейме
df.isna().sum()

name                2
platform            0
year_released     275
genre               2
na_sales            0
eu_sales            6
jp_sales            4
other_sales         0
critic_score     8714
user_score       9268
rating           6871
dtype: int64

In [13]:
# Подсчитываем процент строк с пропусками
df.isna().sum() / len(df) * 100

name              0.011795
platform          0.000000
year_released     1.621845
genre             0.011795
na_sales          0.000000
eu_sales          0.035386
jp_sales          0.023590
other_sales       0.000000
critic_score     51.391838
user_score       54.659118
rating           40.522529
dtype: float64

Незначительный объем данных отсутствует в столбцах `name`, `year_released`, `genre`, `eu_sales`, `jp_sales` - их можно безболезненно удалить.

В столбцах с оценками критиков, пользовательскими оценками и рейтингом отсутствует 40-50% значений. Это может быть связано со следующими факторами:
- Не все игры получают рецензии и оценки критиков
- Оценок может быть слишком мало, чтобы сформировать средний показатель по конкретной игре, особенно если проект небольшой или только выпустившийся
- Платформы для сбора оценок и рецензий появились позже многих старых игр
- Данные могли пропасть при объединении их из разных источников
- Для некоторых некрупных релизов возрастной рейтинг может не присваиваться

In [14]:
# Проверим гипотезу о связи пропусков с годом релиза
df_copy = df.copy()
df_copy['no_critic_score'] = df_copy['critic_score'].isna()
df_copy['no_user_score'] = df_copy['user_score'].isna()
df_copy['no_rating'] = df_copy['rating'].isna()
df_copy.groupby('year_released')[['no_critic_score', 'no_user_score', 'no_rating']].mean().sort_values('no_critic_score')

,no_critic_score,no_user_score,no_rating
year_released,,,
2002,0.244338,0.441001,0.210965
2003,0.245881,0.334601,0.209125
2004,0.264591,0.360571,0.216602
2005,0.303158,0.378947,0.247368
2001,0.321792,0.464358,0.295316
2006,0.386275,0.454902,0.329412
2007,0.419540,0.474548,0.312808
2013,0.496377,0.440217,0.418478
2008,0.498270,0.536332,0.312803


Действительно, до 2000 года львиная доля игр не получала достаточного количества оценок критиков и пользовательских оценок, а также им часто не определяли рейтинг.

In [15]:
# Проверим гипотезу о связи пропусков с платформой, на которой выходила игра
df_copy.groupby('platform', observed=True)[['no_critic_score', 'no_user_score', 'no_rating']].mean().sort_values('no_critic_score')

,no_critic_score,no_user_score,no_rating
platform,,,
XB,0.120381,0.289631,0.109654
GC,0.195382,0.342806,0.158082
PC,0.266667,0.210101,0.205051
X360,0.273224,0.235753,0.166276
XOne,0.318725,0.266932,0.243028
PS4,0.362025,0.349367,0.351899
PS3,0.385978,0.343911,0.287085
WiiU,0.387755,0.319728,0.285714
PS2,0.399726,0.424852,0.314756


Можно заметить, что для определенных платформ пропусков в данных заметно больше, чем для других. А почти для половины они отсутствуют полностью.

Вероятнее всего, для пропусков в `critic_score`, `user_score` и `rating` характерен механизм MAR - пропуски зависят от наблюдаемых переменных (например, платформы, года выпуска и, вероятно, популярности игры).

**Исходя из поставленных задач в проекте, выбрана следующая стратегия работы с пропусками:**
- **Не удалять строки.** Удаление приведёт к сильной потере данных и занижению объема выпущенных игр.
- **Не заполнять средним/медианой.** Для категоризации подстановка среднего значения искусственно надует среднюю категорию и исказит доли групп.
- **Заменить пропуски в оценках на -1 и создать категорию "Нет оценки".** Это самый корректный вариант для задачи категоризации, так как все данные будут сохранены и неопределенность будет честно отражена.
- **Оставить пропуски в столбце с рейтингом.** Эти данные не понадобится дополнительно обрабатывать, поэтому данные пропуски можно проигнорировать.

In [16]:
# Удалим пропуски в столбцах 'name', 'year_released' и 'genre'
df = df.dropna(subset=['name', 'year_released', 'genre', 'eu_sales', 'jp_sales'])

In [17]:
# Заменим пропуски в столдбцах 'critic_score' и 'user_score' на значение по умолчанию
df['critic_score'] = df['critic_score'].fillna(-1)
df['user_score'] = df['user_score'].fillna(-1)

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16669 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   name           16669 non-null  object  
 1   platform       16669 non-null  category
 2   year_released  16669 non-null  Int16   
 3   genre          16669 non-null  category
 4   na_sales       16669 non-null  float32 
 5   eu_sales       16669 non-null  float32 
 6   jp_sales       16669 non-null  float32 
 7   other_sales    16669 non-null  float32 
 8   critic_score   16669 non-null  float32 
 9   user_score     16669 non-null  float32 
 10  rating         9892 non-null   category
dtypes: Int16(1), category(3), float32(6), object(1)
memory usage: 751.2+ KB


### Работа с дубликатами

In [19]:
# Проверим наличие неявных дубликатов среди платформ
df['platform'].value_counts()

platform
PS2     2153
DS      2144
PS3     1330
Wii     1305
X360    1246
PSP     1211
PS      1208
PC       972
GBA      825
XB       818
GC       549
3DS      522
PSV      434
PS4      395
N64      320
XOne     250
SNES     241
SAT      174
WiiU     147
2600     118
NES      100
GB        97
DC        52
GEN       27
NG        12
SCD        6
WS         6
3DO        3
TG16       2
PCFX       1
GG         1
Name: count, dtype: int64

In [20]:
# Проверим наличие неявных дубликатов среди жанров игр
df['genre'].value_counts()

genre
Action          3337
Sports          2325
Misc            1738
Role-Playing    1493
Shooter         1313
Adventure       1309
Racing          1244
Platform         889
Simulation       866
Fighting         843
Strategy         679
Puzzle           577
ACTION            12
SPORTS             7
ROLE-PLAYING       6
FIGHTING           6
RACING             6
SHOOTER            5
ADVENTURE          4
PLATFORM           3
MISC               3
PUZZLE             2
SIMULATION         1
STRATEGY           1
Name: count, dtype: int64

Видно, что среди жанров присутствуют дубликаты с разным регистром. Приведем их к одному виду.

In [21]:
# Приведем данные к одному виду
df['genre'] = df['genre'].str.lower()

In [22]:
# Проверим наличие неявных дубликатов среди названий игр
df['name'].value_counts()

name
Need for Speed: Most Wanted              12
Terraria                                  9
FIFA 14                                   9
LEGO Marvel Super Heroes                  9
Monopoly                                  9
                                         ..
Musou Orochi Z                            1
Tetris 2 (weekly jp sales)                1
Ratchet & Clank: Full Frontal Assault     1
Mega Man & Bass                           1
Haitaka no Psychedelica                   1
Name: count, Length: 11422, dtype: int64

In [23]:
# Проверим причину возникновения дубликатов
df[df['name'] == 'Need for Speed: Most Wanted']

,name,platform,year_released,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
253,Need for Speed: Most Wanted,PS2,2005,racing,2.03,1.79,0.08,0.47,82.0,9.1,T
525,Need for Speed: Most Wanted,PS3,2012,racing,0.71,1.46,0.06,0.58,-1.0,-1.0,NaN
1197,Need for Speed: Most Wanted,X360,2012,racing,0.62,0.78,0.01,0.15,83.0,8.5,T
1605,Need for Speed: Most Wanted,X360,2005,racing,1.00,0.13,0.02,0.10,83.0,8.5,T
2017,Need for Speed: Most Wanted,XB,2005,racing,0.53,0.46,0.00,0.05,83.0,8.8,T
2067,Need for Speed: Most Wanted,PSV,2012,racing,0.33,0.45,0.01,0.22,-1.0,-1.0,NaN
3619,Need for Speed: Most Wanted,GC,2005,racing,0.43,0.11,0.00,0.02,80.0,9.1,T
6051,Need for Speed: Most Wanted,PC,2005,racing,0.02,0.23,0.00,0.04,82.0,8.5,T
6356,Need for Speed: Most Wanted,WiiU,2013,racing,0.13,0.12,0.00,0.02,-1.0,-1.0,NaN
6493,Need for Speed: Most Wanted,DS,2005,racing,0.24,0.01,0.00,0.02,45.0,6.1,E


Дубликаты возникают в связи с релизами игры на отдельных платформах и перезапусками позже. Такие дубликаты можно пропустить, так как фактически это разные релизы.

In [24]:
# Проверим сколько в данных явных дубликатов
df['name'] = df['name'].str.lower()
df['name'] = df['name'].str.strip()
df.duplicated().sum()

np.int64(235)

In [25]:
# Удалим явные дубликаты
df = df.drop_duplicates()

In [26]:
# Посчитаем сколько строк было удалено в результате предобработки
deleted_rows = initial_rows - df.shape[0]
display(deleted_rows)
display(round(deleted_rows / initial_rows * 100, 2))

522

3.08

Удалено 522 строки или же 3.08% от изначального объема данных

### Фильтрация данных

In [27]:
# Отберем данные с 2000 по 2013 год включительно
df_actual = df[df['year_released'].between(2000, 2013)].copy()

### Категоризация данных

In [28]:
# Разделим все игры по оценками пользователей
df_actual['us_score_lvl'] = pd.cut(df['user_score'], bins=[-2, -1, 2, 7, 10], labels=['нет оценки', 'низкая оценка', 'средняя оценка', 'высокая оценка'])

In [29]:
# Разделим все игры по оценками критиков
df_actual['cr_score_lvl'] = pd.cut(df['critic_score'], bins=[-2, -1, 29, 79, 100], labels=['нет оценки', 'низкая оценка', 'средняя оценка', 'высокая оценка'])

## Анализ данных

### Анализ игровых платформ

In [30]:
# Агрегируем суммарное количество релизов по каждой платформе
platform_counts = df_actual['platform'].value_counts()

In [31]:
# Отберем топ-7 платформ по релизам
platform_counts.head(7)

platform
PS2     2126
DS      2117
Wii     1275
PSP     1179
X360    1118
PS3     1087
GBA      810
Name: count, dtype: int64

In [32]:
# Соберем продажи с разных регионов в один показатель
sales_cols = ['na_sales', 'eu_sales', 'jp_sales', 'other_sales']
df_actual['global_sales'] = df_actual[sales_cols].sum(axis=1)

In [33]:
# Агрегируем совокупные продажи по каждой платформе
platform_sales = df_actual.groupby(['platform'], observed=True)['global_sales'].sum().sort_values(ascending=False)

In [34]:
# Отберем топ-7 платформ по продажам
platform_sales.head(7)

platform
PS2     1231.920044
X360     905.849976
Wii      886.109985
PS3      863.159973
DS       796.989990
GBA      311.480011
PSP      288.750000
Name: global_sales, dtype: float32

**На основе полученных результатов можно сделать следующие выводы:**
- PS2 и DS, делящие первое место, практически равны по количеству релизов и заметно опережают остальные платформы — это были массовые платформы с очень высокой активностью разработчиков.
- Wii, несмотря на более узкую нишу (семейные игры с отслеживанием движения в пространстве), тоже получила большую библиотеку.
- Разрыв между топ‑2 и остальными значителен: у Wii почти на 900 игр меньше, чем у PS2/DS.
- PS2 — абсолютный лидер не только по количеству игр, но и по суммарным продажам. Это подтверждает её статус самой успешной консоли в анализируемый период времени.
- DS, несмотря на второе место по количеству релизов, по суммарным продажам находится на пятом месте. Это может быть связано с более низкой средней стоимостью игр, более узкой географией или тем, что многие игры на портативной платформе имели меньший тираж на единицу.
- X360 по количеству игр лишь на пятом месте, но по продажам — на втором. Это может говорить о болешей успешности и популярности проектов.
- В обоих топах сразу 3 платформы (PS2, PS3 и PSP) представлены одной компанией — Sony .

### Анализ жанров игр

In [35]:
# Агрегируем суммарное количество релизов по каждому жанру
genre_counts = df_actual['genre'].value_counts()

In [36]:
# Отберем топ-5 жанров по релизам
genre_counts.head()

genre
action          2456
sports          1818
misc            1482
role-playing    1079
adventure       1009
Name: count, dtype: int64

In [37]:
# Агрегируем совокупные продажи по каждому жанру
genre_sales = df_actual.groupby(['genre'], observed=True)['global_sales'].sum().sort_values(ascending=False)

In [38]:
# Отберем топ-5 жанров по продажам
genre_sales.head()

genre
action          1318.800049
sports          1021.440002
shooter          741.760010
misc             676.279968
role-playing     629.779968
Name: global_sales, dtype: float32

**На основе полученных результатов можно сделать следующие выводы:**
- Жанр action — лидер как по количеству релизов, так и по суммарным продажам. Это говорит о высокой активности разработчиков в этом сегменте и, вероятно, о меньшем среднем цикле разработки или более широкой целевой аудитории, а также о коммерческой успешности.
- Sports и misc занимают второе и третье места: спортивные игры часто выпускают обновленные версии и новые части, а жанр misc представляет собой смесь других жанров, поэтому является массовым.
- Role‑playing и adventure — более нишевые по количеству релизов.
- Жанр shooter — не вошёл в топ‑5 по количеству релизов, но занял третье место по продажам. Это признак высокой эффективности: относительно немного игр в релизе, но каждая из них имеет большой тираж.
- Жанр role-playing остает по количеству релизов от лидера почти в 2,5 раза, но по продажам лишь в 2 раза: RPG‑проекты часто имеют более длинный цикл разработки и более высокую стоимость.

### Анализ данных по годам

In [39]:
# Агрегируем суммарное количество релизов по каждому году
year_counts = df_actual['year_released'].value_counts()

In [40]:
# Изучим динамику по годам
year_counts.sort_index()

year_released
2000     350
2001     482
2002     829
2003     775
2004     760
2005     939
2006    1005
2007    1194
2008    1425
2009    1425
2010    1255
2011    1136
2012     653
2013     544
Name: count, dtype: Int64

In [41]:
# Получаем индекс строки с максимальными продажами в каждом году
idx_max_sales = df_actual.groupby('year_released')['global_sales'].idxmax()

# Формируем таблицу лидеров по годам
top_games_by_year = df_actual.loc[idx_max_sales, ['year_released', 'name', 'genre', 'platform', 'global_sales']].reset_index(drop=True)

In [42]:
# Выедем результат
top_games_by_year

,year_released,name,genre,platform,global_sales
0,2000,pokémon crystal version,role-playing,GB,6.389999
1,2001,gran turismo 3: a-spec,racing,PS2,14.970000
2,2002,grand theft auto: vice city,action,PS2,16.150000
3,2003,need for speed underground,racing,PS2,7.200000
4,2004,grand theft auto: san andreas,action,PS2,20.809999
5,2005,nintendogs,simulation,DS,24.670000
6,2006,wii sports,sports,Wii,82.539993
7,2007,wii fit,sports,Wii,22.700001
8,2008,mario kart wii,racing,Wii,35.520000
9,2009,wii sports resort,sports,Wii,32.770000


**На основе полученных результатов можно сделать следующие выводы:**
- 2000–2002: резкий рост с 350 до 829 релизов (+137% за 2 года). Вероятно это связано с расцветом платформы PS2.
- 2003–2007: стабильный рост от 775 до 1194 релизов. Несмотря на небольшие колебания, тренд остается восходящим.
- 2008–2009: пик активности — по 1425 релизов в оба года. Это максимум за весь период.
- Продажи лидера года стабильно росли с 6,4 млн копий в 2000 году до пика в 82,5 млн в 2006 году. Это почти 13‑кратный рост за 6 лет.
- В топ‑продажах представлены разные жанры, но в основном лидерами становятся игры, ориентированные на широкую аудиторию (спортивные и экшен игры)

### Анализ по регионам

In [43]:
# Суммируем продажи по жанрам для каждого региона
genre_na = df_actual.groupby('genre', observed=False)['na_sales'].sum()
genre_eu = df_actual.groupby('genre', observed=False)['eu_sales'].sum()
genre_jp = df_actual.groupby('genre', observed=False)['jp_sales'].sum()
genre_other = df_actual.groupby('genre', observed=False)['other_sales'].sum()

In [44]:
# Суммируем продажи по платформам для каждого региона
platform_na = df_actual.groupby('platform', observed=False)['na_sales'].sum()
platform_eu = df_actual.groupby('platform', observed=False)['eu_sales'].sum()
platform_jp = df_actual.groupby('platform', observed=False)['jp_sales'].sum()
platform_other = df_actual.groupby('platform', observed=False)['other_sales'].sum()

In [45]:
# Для каждого региона выбираем жанр и платформу с максимальной суммой продаж:
def get_top(series):
    return series.idxmax()

top_genre_na = get_top(genre_na)
top_platform_na = get_top(platform_na)

top_genre_eu = get_top(genre_eu)
top_platform_eu = get_top(platform_eu)

top_genre_jp = get_top(genre_jp)
top_platform_jp = get_top(platform_jp)

top_genre_other = get_top(genre_other)
top_platform_other = get_top(platform_other)

In [46]:
# Формируем итоговую таблицу для анализа региональных предпочтений
regional_leaders = pd.DataFrame({
    'region': ['NA', 'EU', 'JP', 'Other'],
    'top_genre': [top_genre_na, top_genre_eu, top_genre_jp, top_genre_other],
    'top_platform': [top_platform_na, top_platform_eu, top_platform_jp, top_platform_other]
}).set_index('region')

In [47]:
# Выводим результат
regional_leaders

,top_genre,top_platform
region,,
NA,action,PS2
EU,action,PS2
JP,role-playing,DS
Other,action,PS2


**На основе полученных результатов можно сделать следующие выводы:**
- В большинстве регионов самым популярным жанром игр является экшен, а платформой — PS2. Это неудивительно, ведь рассматреваемый период охватывает время расцвета данной консоли.
- Среди 4 регионов явно выделяется лишь Япония — только там наиболее популярным оказался жанр игр RPG, а самая популярная платформа DS — карманная игровая консоль.

### Анализ игр жанра RPG

In [48]:
# Отберем игры этого жанра
rpg_df = df_actual[df_actual['genre'] == 'role-playing'].copy()

In [49]:
# Отберем топ-10 игр этого жанра по продажам
top_10_rpg = rpg_df.sort_values(by='global_sales', ascending=False).head(10)

In [50]:
# В итоговую таблицу включим суммарные продажи и рейтинг игры
result_table = top_10_rpg[['name', 'global_sales', 'rating']].reset_index(drop=True)

In [51]:
# Выведем результат
result_table

,name,global_sales,rating
0,pokemon diamond/pokemon pearl,18.240002,NaN
1,pokemon ruby/pokemon sapphire,15.840000,NaN
2,pokemon black/pokemon white,15.130000,NaN
3,pokemon x/pokemon y,14.599999,NaN
4,pokemon firered/pokemon leafgreen,10.490001,NaN
5,the elder scrolls v: skyrim,8.790000,M
6,pokemon black 2/pokemon white 2,8.060000,NaN
7,final fantasy x,8.040000,T
8,pokémon platinum version,7.710000,E
9,pokémon emerald version,6.420000,E


**На основе полученных результатов можно сделать следующие выводы:**
- В топ‑10 сразу 8 позиций занимают игры серии Pokemon. Это показывает исключительную коммерческую силу франшизы в жанре RPG.
- Большинство игр Pokemon в таблице не имеют указанного возрастного рейтинга, возможно возрастные рейтинги в Японии отличались от общепринятых в рассмотренный период. Тем неменее, исторически эти игры получают детский рейтинг (E).
- the elder scrolls v: skyrim — единственная игра в топе с рейтингом для взрослых (M). Её успех демонстрирует, что и взрослые RPG могут достигать очень высоких тиражей, хотя и уступают массовым сериям.
- final fantasy x — единственная игра в топе с подростковым рейтингом (T). Его позиция подтверждает устойчивый спрос на традиционные JRPG даже вне франшизы Pokemon.

### Анализ оценок пользователей и критиков

In [52]:
# Проанализируем распределение оценок
display(df_actual['us_score_lvl'].value_counts(normalize=True))
display(df_actual['cr_score_lvl'].value_counts(normalize=True))

us_score_lvl
нет оценки        0.492953
высокая оценка    0.319292
средняя оценка    0.184075
низкая оценка     0.003680
Name: proportion, dtype: float64

cr_score_lvl
нет оценки        0.439242
средняя оценка    0.424444
высокая оценка    0.132008
низкая оценка     0.004306
Name: proportion, dtype: float64

**На основе полученных результатов можно сделать следующие выводы:**
- У большинства игр оценки отсутствуют
- Пользовательские оценки почти в 1/3 случаев оказываются высокими
- Оценки критиков в большиснтве случаев оказываются средними

In [53]:
# Сравним оценки пользователей и критиков
pd.crosstab(df_actual['us_score_lvl'], df_actual['cr_score_lvl'], normalize='index')

cr_score_lvl,нет оценки,низкая оценка,средняя оценка,высокая оценка
us_score_lvl,,,,
нет оценки,0.831480,0.001112,0.162166,0.005241
низкая оценка,0.340426,0.127660,0.531915,0.000000
средняя оценка,0.073160,0.017439,0.824330,0.085070
высокая оценка,0.045856,0.000245,0.597597,0.356302


**На основе полученных результатов можно сделать следующие выводы:**
- Когда игроки ставят низкую оценку, критики чаще всего они либо не оценивают игру, либо дают среднюю оценку. Это может говорить о том, что низкие пользовательские оценки часто связаны с малозаметными проектами.
- Если игроки ставят среднюю оценку, с вероятностью более 80% критики тоже дадут среднюю. На среднем уровне мнения игроков и критиков наиболее согласованы.
- Даже если игроки высоко оценили игру, критики чаще дают среднюю оценку (почти 60%), а высокую — лишь в трети случаев. Это может указывать на более строгие критерии критиков или на то, что народная любовь не всегда совпадает с профессиональной оценкой.

In [54]:
# Оценим связь оценок с продажами
sales_by_us_score = df_actual.groupby('us_score_lvl', observed=False)['global_sales'].sum()
sales_by_cr_score = df_actual.groupby('cr_score_lvl', observed=False)['global_sales'].sum()

In [55]:
# Выедем результат по пользовательским оценкам
sales_by_us_score

us_score_lvl
нет оценки        1566.959961
низкая оценка       10.120000
средняя оценка    1273.219971
высокая оценка    3563.000000
Name: global_sales, dtype: float32

In [56]:
# Выедем результат по  оценкам критиков
sales_by_cr_score

cr_score_lvl
нет оценки        1509.219971
низкая оценка       12.350000
средняя оценка    2361.080078
высокая оценка    2530.649902
Name: global_sales, dtype: float32

**На основе полученных результатов можно сделать следующие выводы:**
- Игры, получившие высокую оценку от пользователей, суммарно продали более чем в 2,8 раза больше копий, чем игры со средней оценкой, и почти в 350 раз больше, чем низко оценённые. Пользовательская оценка очень сильно влияет на коммерческий успех игры.
- В отличие от пользовательских оценок, у критиков игры имеющие средние и высокие оценки имеют сопоставимые объёмы продаж (2361 против 2531 млн). Это говорит о том, что игры, получившие сдержанную профессиональную оценку, всё равно могут быть коммерчески успешными.
- Игры в категории с низкой оценкой и у пользователей, и у критиков практически не вносят вклад в общие продажи.
- Игры без оценок пользователей имеют больше продаж, чем игры со средними оценками. У критиков этот сегмент также значим. Вероятно туда входят нишевые и старые игры, у которых нет оценок в современных базах.

## Выводы

**В данном проекте была проделана следующая работа:**
- Проведен разведочный анализ данных
- Осуществлена предобработка данных
- Отобран срез данных с 2000 по 2013 год включительно
- Добавлены новые столбцы: `global_sales`, `us_score_lvl` и `cr_score_lvl`
- Проанализированы данные по платформам, жанрам, годам, оценкам и внутри жанра RPG

**По итогам всего анализа можно сформировать следующие выводы:**
- Самая популярная и комерчески успешная платформа — это консоль `PS2` компании Sony, в топ по популярности вошли и другие консоли этой же компании.
- Наиболее популярны как по релизам, так и по продажам жанры игр `action` и `sports`.
- До 2008 - 2009 года количество ежегодных релизов активно росло (за 8 лет более чем в 4 раза), достигнув пика в 1425 релизов в оба года, далее заметен спад.
- В большиснстве регионов наиболее популярный жанр и платформа совпадают с результатами выше, но в Японии более популярным оказался жанр `RPG`, а платформа `DS`.
- Среди игр жанра `RPG` 8 из 10 позиций топа заняли игры серии Pokemon — здесь влияет специфика японского региона, так как эта серия игр принадлежит японской компании.
- Обычные пользователи чаще критиков ставят высокие оценки играм. При этом игры, которые пользователи оценивают высоко, продаются заметно чаще игр с более низкой оценкой. В то же время игры со средней и высокой оценкой критиков продаются одинаково. Это означает, что народная любовь сильнее влияет на коммерческий успех игрового проекта, нежели мнение критиков.